In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

In [4]:
# data_one contains the list of dic from the trainset
with open('../data/trainset/training_set_task2.txt', 'r', encoding='utf-8') as file:
    data_one = json.load(file)

# data_two contains the list of dic from the devset
with open('../data/trainset/dev_set_task2.txt', 'r', encoding='utf-8') as file:
    data_two = json.load(file)

# data_three contains the list of dic from the devset
with open('../data/testset/test_set_task2.txt', 'r', encoding='utf-8') as file:
    data_three = json.load(file)

# Both datasets are flattened/normalised to make comparison 
df_train = pd.json_normalize(data_one, record_path=['labels'], meta=['id', 'text'])
df_dev = pd.json_normalize(data_two, record_path=['labels'], meta=['id', 'text'])
df_test = pd.json_normalize(data_three, record_path=['labels'], meta=['id', 'text'])

# Functions

In [4]:
def tokenisation(text):
    tokens = re.split(r'\s+', text.strip())
    return tokens

In [5]:
def numerical_mask(text):
    token_count = 0
    numerical_format = []
    done= False
    for char in text:
        if char == ' ':
            numerical_format.append(-1)
            if not done:
                token_count += 1
                done = True
            #token_count += 1
        elif char == '\n':
            numerical_format.append(-2)
            if not done:
                token_count += 1
                done = True
        else:
            numerical_format.append(token_count)
            done = False

    return numerical_format

In [6]:
def char_to_word_level(numerical_format,spans):
    new_spans = []
    for span in spans:
        #print(span)
        if numerical_format[span['start']] != -1 and numerical_format[span['start']] != -2:
            start = numerical_format[span['start']]
            #print(start)
        else:
            for i in range (span['start'],len(numerical_format)):
                if numerical_format[i] != -1 and numerical_format[i] != -2:
                    start = numerical_format[i]
                    break

        if numerical_format[(span['end'] - 1)] != -1 and numerical_format[(span['end'] - 1)] != -2:
            end = numerical_format[(span['end']-1)]
            #print(end)
        else:
            for i in range((span['end'] - 1),0, -1):
                if numerical_format[i] != -1 and numerical_format[i] != -2:
                    end = numerical_format[i]
                    break

        new_spans.append({'start':start,'end': end, 'technique':span['technique'], 'text_fragment': span['text_fragment']})

    return new_spans

In [7]:

def tokenise_text_and_spans_char_to_word_level(data):
    new_data = []

    for i in range(len(data)):
        #print(i)
        if(data[i]['labels']):
            text_numerical_mask = numerical_mask(data[i]['text'])
            new_spans = char_to_word_level(text_numerical_mask, data[i]['labels'])
            new_data.append({'id':data[i]['id'],'text':tokenisation(data[i]['text']),'labels':new_spans})
        else:
            new_data.append({'id':data[i]['id'],'text':tokenisation(data[i]['text']),'labels':data[i]['labels']})

    return new_data

In [8]:
## Function which identifies independent spans
def find_independent_spans(labels,df_train):
    # Sort spans by their end position
    labels.sort(key=lambda x: x['end'])

    total_overlapping_spans = 0
    independent_spans = []
    last_end = -1
    techniques = df_train['technique'].unique()
    techniques_dict = {technique: 0 for technique in techniques}

    for label in labels:
        if label['start'] > last_end:
            independent_spans.append(label)
            last_end = label['end']
        else:
            techniques_dict[label['technique']] += 1
            total_overlapping_spans += 1


    return independent_spans, techniques_dict, total_overlapping_spans

In [9]:
# Function to tokenize and align labels
def tokenize_and_align_labels(short_dataset, list_name,label_all_tokens=True ):
    # Tokenize the input tokens with truncation and word splitting enabled
    tokenized_inputs = tokenizer(short_dataset, truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(list_name):
        # Get the word IDs for the i-th example in the batch
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word ID that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    # Add the aligned labels to the tokenized inputs
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [10]:
def removing_overlapping_enclosing_spans(data, df):
    new_data = []

    for i in range(0,len(data)):
        if(data[i]['labels']):
            temp_spans, _, _ = find_independent_spans(data[i]['labels'],df)
            new_data.append({'id':data[i]['id'],'text':data[i]['text'],'labels':temp_spans})
        else:
            new_data.append({'id':data[i]['id'],'text':data[i]['text'],'labels':data[i]['labels']})

    return new_data

In [11]:
def labels_to_list_and_tokens_in_list(new_info):
    data_preprocessed_x = []
    data_preprocessed_y = []

    for text in new_info:
        data_preprocessed_x.append(text['text'])
        temp = [0 for i in range(len(text['text']))]

        for span in text['labels']:
            temp[span['start']] = 1
            for i in range(span['start'] + 1, span['end'] + 1):
                temp[i] = 2

        data_preprocessed_y.append(temp)

    return data_preprocessed_x, data_preprocessed_y

In [12]:
def convert_to_start_end_labels(dataset):
    tokenised_text = []
    span_start = []
    span_end = []

    for text in dataset:
        tokenised_text.append(text['text'])

        temp_start = [0 for i in range(len(text['text']))]
        temp_end = [0 for i in range(len(text['text']))]

        for span in text['labels']:
            temp_start[span['start']] = 1
            temp_end[span['end']] = 1


        span_start.append(temp_start)
        span_end.append(temp_end)


    return tokenised_text, span_start, span_end

In [13]:
# Function to tokenize and align labels
def tokenize_and_align_labels_for_start_end(dataset, start_list, end_list,label_all_tokens=True ):
    # Tokenize the input tokens with truncation and word splitting enabled
    tokenized_inputs = tokenizer(dataset, truncation=True, is_split_into_words=True)

    start_labels = []
    for i, label in enumerate(start_list):
        # Get the word IDs for the i-th example in the batch
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word ID that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        start_labels.append(label_ids)

    end_labels = []
    for i, label in enumerate(end_list):
        # Get the word IDs for the i-th example in the batch
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word ID that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        end_labels.append(label_ids)

    # Add the aligned labels to the tokenized inputs
    tokenized_inputs["span_start"] = start_labels
    tokenized_inputs["span_end"] = end_labels
    return tokenized_inputs